# Kalman Historical V3.4 — Allocator Regime Gate

V3.3에서 확인된 `HYBRID_USV2_KRV3_BTCV2`를 고정하고, 최근 구간에서 Max Sharpe의 높은 변동성을 제어하는 allocator gate를 검증합니다.

### Fixed base
- Sleeve: US V2 / KR V3 / BTC V2
- Lookback: 180 days
- Rebalance: Monthly
- Overlay cost: 10 bps per traded notional

### Gate family
- Static Equal Weight benchmark
- Static Max Sharpe benchmark
- Fallback -> Equal Weight
- Predicted-vol cap: 1.10 / 1.20 / 1.30 / 1.50 × Equal Weight predicted vol
- Risk-cap gate blends EW and Max Sharpe instead of hard-switching when possible

### Anti-lookahead selection
- Gate selection uses DEV only
- V3.3 RECENT start is locked as holdout boundary
- Recent holdout is never used to select the gate

### Pass criteria
- Recent Sharpe >= static Max Sharpe and >= 95% of Equal Weight Sharpe
- Recent MDD no worse than static Max Sharpe
- Retain >=85% of best static recent return
- Full-period risk-adjusted profile must remain acceptable

**Research only / Toss OFF / Neon write OFF / LIVE OFF**


In [ ]:
from google.colab import drive
import json
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

PINNED_SHA = "d0d11213267718e54f2a2b93c664da128f6d1d4d"
SOURCE_BRANCH = "feature/historical-v3-4-allocator-regime-gate-20260913"

V2_TAG = "20260913_nested_v2_001"
V3_TAG = "20260913_return_regime_v3_001"
V33_TAG = "20260913_v3_3_robustness_hybrid_001"
V34_TAG = "20260913_v3_4_allocator_regime_gate_001"

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")


def run(cmd, *, cwd=None, log_path=None):
    args = [str(x) for x in cmd]
    header = "\n$ " + " ".join(args) + "\n"
    print(header, end="")
    chunks = [header]
    proc = subprocess.Popen(
        args,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        chunks.append(line)
        print(line, end="")
    rc = proc.wait()
    output = "".join(chunks)
    if log_path is not None:
        with Path(log_path).open("a", encoding="utf-8") as fh:
            fh.write(output)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args, output=output)
    return output


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8",
    )
    tmp.replace(path)


def find_model_root(root):
    candidates = [
        root / "Market_Model_V2",
        root / "Kalman" / "Market_Model_V2",
        root / "kalman" / "Market_Model_V2",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Market_Model_V2 root not found")


model_root = find_model_root(DRIVE_ROOT)
v2_root = model_root / "historical_quant_2017_v2_candidate" / V2_TAG
v3_root = model_root / "historical_quant_2017_v3_candidate" / V3_TAG
v33_root = model_root / "historical_quant_2017_v3_3_robustness" / V33_TAG
v33_summary = v33_root / "v3_3_summary.json"
out_root = model_root / "historical_quant_2017_v3_4_allocator_gate" / V34_TAG
out_root.mkdir(parents=True, exist_ok=True)

status_path = out_root / "v3_4_colab_status.json"
full_log = out_root / "v3_4_full_log.txt"
failure_path = out_root / "v3_4_failure.json"
full_log.write_text("", encoding="utf-8")


def status(state, phase, **extra):
    write_json(
        status_path,
        {
            "status": state,
            "phase": phase,
            "updated_at": datetime.now().astimezone().isoformat(),
            "pinned_sha": PINNED_SHA,
            "v2_tag": V2_TAG,
            "v3_tag": V3_TAG,
            "v3_3_tag": V33_TAG,
            "v3_4_tag": V34_TAG,
            "research_only": True,
            "live_execution": False,
            "toss_execution": False,
            "neon_write": False,
            **extra,
        },
    )


try:
    for required in (
        v2_root / "historical_v2_candidate_summary.json",
        v3_root / "historical_v3_candidate_summary.json",
        v33_summary,
    ):
        assert required.exists(), required

    v33_payload = json.loads(v33_summary.read_text(encoding="utf-8"))
    assert v33_payload.get("status") == "COMPLETE", v33_payload
    assert v33_payload.get("experiment_status") == "READY", v33_payload
    assert (
        v33_payload.get("recommendation", {}).get("recommended_source_set")
        == "HYBRID_USV2_KRV3_BTCV2"
    ), v33_payload.get("recommendation")

    repo = Path("/content/Codex")
    if repo.exists():
        shutil.rmtree(repo)

    status("RUNNING", "CLONE")
    run(
        [
            "git",
            "clone",
            "--branch",
            SOURCE_BRANCH,
            "https://github.com/kimtk94/Codex.git",
            repo,
        ],
        log_path=full_log,
    )
    run(
        ["git", "-C", repo, "checkout", "--detach", PINNED_SHA],
        log_path=full_log,
    )
    checked = subprocess.check_output(
        ["git", "-C", repo, "rev-parse", "HEAD"],
        text=True,
    ).strip()
    assert checked == PINNED_SHA, (checked, PINNED_SHA)

    app = repo / "kalman-toss-gateway"
    module = app / "research" / "quant_stack" / "historical_v3_4_allocator_regime_gate.py"
    test_file = app / "tests" / "test_historical_v3_4_allocator_regime_gate.py"
    assert module.exists(), module
    assert test_file.exists(), test_file

    status("RUNNING", "ISOLATED_ENV")
    if shutil.which("uv") is None:
        run(
            [sys.executable, "-m", "pip", "install", "-q", "uv"],
            log_path=full_log,
        )
    uv = shutil.which("uv")
    assert uv

    venv = Path("/content/.venv-kalman-v3-4")
    if venv.exists():
        shutil.rmtree(venv)
    run([uv, "venv", venv], log_path=full_log)
    vpy = venv / "bin" / "python"

    run(
        [
            uv,
            "pip",
            "install",
            "--python",
            vpy,
            "pandas==3.0.5",
            "numpy==2.4.6",
            "pyarrow",
            "scipy<1.18",
            "scikit-learn",
            "PyPortfolioOpt==1.6.0",
            "pytest",
        ],
        log_path=full_log,
    )
    run(
        [uv, "pip", "check", "--python", vpy],
        log_path=full_log,
    )

    versions = run(
        [
            vpy,
            "-c",
            (
                "import pandas as pd,numpy as np,pypfopt; "
                "print('pandas='+pd.__version__); "
                "print('numpy='+np.__version__); "
                "print('pypfopt='+pypfopt.__version__)"
            ),
        ],
        cwd=app,
        log_path=full_log,
    )

    status("RUNNING", "STATIC_AND_UNIT_TESTS", versions=versions)
    run(
        [vpy, "-m", "py_compile", module, test_file],
        cwd=app,
        log_path=full_log,
    )
    run(
        [
            vpy,
            "-m",
            "pytest",
            "-q",
            "tests/test_historical_v3_4_allocator_regime_gate.py",
        ],
        cwd=app,
        log_path=full_log,
    )

    status("RUNNING", "ALLOCATOR_REGIME_GATE")
    run(
        [
            vpy,
            "-m",
            "research.quant_stack.historical_v3_4_allocator_regime_gate",
            "--v2-root",
            v2_root,
            "--v3-root",
            v3_root,
            "--v3-3-summary",
            v33_summary,
            "--output-dir",
            out_root,
            "--code-sha",
            PINNED_SHA,
        ],
        cwd=app,
        log_path=full_log,
    )

    summary = out_root / "v3_4_summary.json"
    metrics = out_root / "v3_4_gate_metrics.csv"
    audit = out_root / "v3_4_gate_audit.csv"
    dev_ranking = out_root / "v3_4_dev_ranking.csv"

    for required in (summary, metrics, audit, dev_ranking):
        assert required.exists(), required

    payload = json.loads(summary.read_text(encoding="utf-8"))

    snap = out_root / "code_snapshot"
    snap.mkdir(exist_ok=True)
    shutil.copy2(module, snap / module.name)
    shutil.copy2(test_file, snap / test_file.name)

    status(
        "COMPLETE",
        "DONE",
        experiment_status=payload.get("experiment_status"),
        selected_gate=payload.get("selected_gate"),
        shadow_gate=payload.get("shadow_gate"),
        promotion_recommendation=payload.get("promotion_recommendation"),
        summary=str(summary),
        versions=versions,
    )

    print("\n" + "=" * 96)
    print("KALMAN V3.4 ALLOCATOR REGIME GATE COMPLETE")
    print("=" * 96)
    print(summary.read_text(encoding="utf-8"))

except Exception as exc:
    payload = {
        "status": "FAIL",
        "updated_at": datetime.now().astimezone().isoformat(),
        "pinned_sha": PINNED_SHA,
        "error_type": type(exc).__name__,
        "error": str(exc),
        "child_output": getattr(exc, "output", None),
        "traceback": traceback.format_exc(),
        "full_log": str(full_log),
    }
    write_json(failure_path, payload)
    status(
        "FAIL",
        "FAILED",
        error_type=payload["error_type"],
        error=payload["error"],
        failure_json=str(failure_path),
        full_log=str(full_log),
    )
    print("\nFAILURE SAVED:", failure_path)
    print(failure_path.read_text(encoding="utf-8"))
    raise
